# Batch image censoring with the Puryfi model

This notebook walks through loading the TensorFlow.js model that ships with the Firefox extension and reusing it from Python for offline image batch processing.


## 1. Install the required Python packages

The notebook relies on TensorFlow for inference, TensorFlow.js utilities for converting the browser model, Pillow for image manipulation, and tqdm for progress bars.


In [ ]:
%pip install --quiet tensorflow tensorflowjs pillow matplotlib tqdm


## 2. Import standard-library helpers


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


## 3. Import third-party libraries


In [ ]:
import numpy as np
from PIL import Image, ImageFilter
from tqdm.auto import tqdm

import tensorflow as tf
import tensorflowjs as tfjs


## 4. Define extension and model paths


In [ ]:
BASE_DIR = Path.cwd()
EXTENSION_DIR = BASE_DIR / "releases" / "puryfi-0.8.0.8-anfx"
MODEL_DIR = EXTENSION_DIR / "web_model"
CONVERTED_MODEL_DIR = BASE_DIR / "artifacts" / "tf_saved_model"
CONVERTED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_JSON_PATH = MODEL_DIR / "model.json"


## 5. Record the detection label ordering

The model returns class indices that map to the same label ordering used in the browser extension.


In [ ]:
LABELS: Tuple[str, ...] = (
    "FACEFEMALE",
    "FACEMALE",
    "BELLYEXPOSED",
    "BELLYCOVERED",
    "BUTTOCKSEXPOSED",
    "BUTTOCKSCOVERED",
    "FEMALEBREASTEXPOSED",
    "FEMALEBREASTCOVERED",
    "FEMALEGENITALIAEXPOSED",
    "FEMALEGENITALIACOVERED",
    "MALEGENITALIACOVERED",
    "MALEGENITALIAEXPOSED",
    "MALEBREASTEXPOSED",
    "MALEBREASTCOVERED",
    "FEETCOVERED",
    "FEETEXPOSED",
    "ARMPITSCOVERED",
    "ARMPITSEXPOSED",
    "ANUSCOVERED",
    "ANUSEXPOSED",
    "EYE",
    "MOUTH",
    "NIPPLECOVERED",
    "NIPPLEEXPOSED",
    "HANDCOVERED",
    "HANDEXPOSED",
)


## 6. Convert the TensorFlow.js GraphModel to a SavedModel

This step is required because TensorFlow.js stores the model weights in sharded `.bin` files. The conversion is performed once and cached on disk.


In [ ]:
def convert_tfjs_graph_model(model_json: Path, output_dir: Path, overwrite: bool = False) -> Path:
    """Convert a TensorFlow.js graph model to a TensorFlow SavedModel."""
    if output_dir.exists() and any(output_dir.iterdir()) and not overwrite:
        return output_dir

    if output_dir.exists() and overwrite:
        for child in list(output_dir.iterdir()):
            if child.is_dir():
                import shutil
                shutil.rmtree(child)
            else:
                child.unlink()

    tfjs.converters.convert_tfjs_graph_model(str(model_json), str(output_dir))
    return output_dir


### Run the conversion


In [ ]:
saved_model_dir = convert_tfjs_graph_model(MODEL_JSON_PATH, CONVERTED_MODEL_DIR)
saved_model_dir


## 7. Load the SavedModel and inspect the signatures


In [ ]:
detector = tf.saved_model.load(str(saved_model_dir))
signature_keys = list(detector.signatures.keys())
signature_keys


### Select the serving signature


In [ ]:
serving_key = signature_keys[0]
serving_fn = detector.signatures[serving_key]
serving_fn


### Record the input and output tensor names


In [ ]:
input_signature = serving_fn.structured_input_signature[1]
input_tensor_key = next(iter(input_signature))
output_specs = serving_fn.structured_outputs
output_tensor_keys = list(output_specs.keys())
input_tensor_key, output_tensor_keys


## 8. Helper utilities for preprocessing and inference


In [ ]:
def load_image(path: Path) -> Image.Image:
    """Load an image and force RGB mode."""
    with Image.open(path) as img:
        return img.convert("RGB")


def prepare_input_tensor(image: Image.Image, size: Tuple[int, int] = (320, 320)) -> tf.Tensor:
    """Resize and normalize an image to match the model input."""
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = tf.convert_to_tensor(array)
    tensor = tf.image.resize(tensor, size, method="bilinear", preserve_aspect_ratio=False)
    tensor = tf.expand_dims(tensor, axis=0)
    return tensor


### Run the model and collect raw outputs


In [ ]:
def run_model(image: Image.Image) -> Dict[str, np.ndarray]:
    inputs = {input_tensor_key: prepare_input_tensor(image)}
    outputs = serving_fn(**inputs)
    return {name: tensor.numpy() for name, tensor in outputs.items()}


### Translate model outputs into detection records


In [ ]:
@dataclass
class Detection:
    label_index: int
    label_key: str
    score: float
    box: Tuple[int, int, int, int]


def parse_detections(image: Image.Image, model_outputs: Dict[str, np.ndarray], score_threshold: float = 0.3) -> List[Detection]:
    width, height = image.size
    boxes = model_outputs[output_tensor_keys[0]][0]
    scores = model_outputs[output_tensor_keys[1]][0]
    classes = model_outputs[output_tensor_keys[2]][0].astype(int)
    valid = int(model_outputs[output_tensor_keys[3]][0])

    detections: List[Detection] = []
    for i in range(valid):
        score = float(scores[i])
        if score < score_threshold:
            continue
        label_index = int(classes[i])
        label_key = LABELS[label_index]
        x1, y1, x2, y2 = boxes[i]
        left = max(0, min(int(round(x1 * width)), width))
        top = max(0, min(int(round(y1 * height)), height))
        right = max(0, min(int(round(x2 * width)), width))
        bottom = max(0, min(int(round(y2 * height)), height))
        if right <= left or bottom <= top:
            continue
        detections.append(Detection(label_index, label_key, score, (left, top, right, bottom)))
    return detections


## 9. Censor effect implementations


In [ ]:
def apply_pixelate(region: Image.Image, pixel_size: int = 12) -> Image.Image:
    pixel_size = max(1, pixel_size)
    small = region.resize(
        (max(1, region.width // pixel_size), max(1, region.height // pixel_size)),
        resample=Image.NEAREST,
    )
    return small.resize(region.size, Image.NEAREST)


def apply_blur(region: Image.Image, radius: float = 16.0) -> Image.Image:
    return region.filter(ImageFilter.GaussianBlur(radius=radius))


def apply_solid(region: Image.Image, color: Tuple[int, int, int] = (0, 0, 0)) -> Image.Image:
    return Image.new("RGB", region.size, color)


def apply_censor(
    region: Image.Image,
    mode: str = "pixelate",
    *,
    pixel_size: int = 12,
    blur_radius: float = 16.0,
    color: Tuple[int, int, int] = (0, 0, 0),
) -> Image.Image:
    mode = mode.lower()
    if mode == "pixelate":
        return apply_pixelate(region, pixel_size=pixel_size)
    if mode == "blur":
        return apply_blur(region, radius=blur_radius)
    if mode == "solid":
        return apply_solid(region, color=color)
    raise ValueError(f"Unsupported censor mode: {mode}")


## 10. Apply detections to an image


In [ ]:
def censor_image(
    image: Image.Image,
    detections: Sequence[Detection],
    *,
    allowed_labels: Optional[Iterable[str]] = None,
    mode: str = "pixelate",
    pixel_size: int = 12,
    blur_radius: float = 16.0,
    color: Tuple[int, int, int] = (0, 0, 0),
) -> Image.Image:
    allowed = {label.upper() for label in allowed_labels} if allowed_labels is not None else None
    censored = image.copy()
    for detection in detections:
        if allowed is not None and detection.label_key.upper() not in allowed:
            continue
        left, top, right, bottom = detection.box
        region = censored.crop((left, top, right, bottom))
        censored_region = apply_censor(
            region,
            mode=mode,
            pixel_size=pixel_size,
            blur_radius=blur_radius,
            color=color,
        )
        censored.paste(censored_region, (left, top))
    return censored


## 11. Batch processing driver


In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}


def process_directory(
    input_dir: Path,
    output_dir: Path,
    *,
    censor_mode: str = "pixelate",
    censor_labels: Optional[Iterable[str]] = None,
    score_threshold: float = 0.3,
    pixel_size: int = 12,
    blur_radius: float = 16.0,
    color: Tuple[int, int, int] = (0, 0, 0),
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = [path for path in input_dir.iterdir() if path.suffix.lower() in SUPPORTED_EXTENSIONS]
    for path in tqdm(paths, desc="Processing images"):
        image = load_image(path)
        raw_outputs = run_model(image)
        detections = parse_detections(image, raw_outputs, score_threshold=score_threshold)
        censored = censor_image(
            image,
            detections,
            allowed_labels=censor_labels,
            mode=censor_mode,
            pixel_size=pixel_size,
            blur_radius=blur_radius,
            color=color,
        )
        censored.save(output_dir / path.name)


## 12. Configure an input/output run


In [ ]:
INPUT_DIRECTORY = BASE_DIR / "sample_input"
OUTPUT_DIRECTORY = BASE_DIR / "sample_output"
CENSOR_LABELS = [
    "FEMALEBREASTEXPOSED",
    "FEMALEGENITALIAEXPOSED",
    "MALEGENITALIAEXPOSED",
]
CENSOR_MODE = "pixelate"
SCORE_THRESHOLD = 0.3


### Execute the batch processor


In [ ]:
process_directory(
    INPUT_DIRECTORY,
    OUTPUT_DIRECTORY,
    censor_mode=CENSOR_MODE,
    censor_labels=CENSOR_LABELS,
    score_threshold=SCORE_THRESHOLD,
    pixel_size=14,
    blur_radius=18.0,
    color=(0, 0, 0),
)


## 13. Optional: Preview a processed image


In [ ]:
from IPython.display import display


def preview_output_image(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image.open(path))

# Example usage (uncomment after running the processor):
# preview_output_image(OUTPUT_DIRECTORY / "example.jpg")
